# 002 - Text Embeddings & Semantic Search (RAG Practice)
Plantilla práctica para practicar chunking, generación de embeddings con Voyage AI, similitud coseno y recuperación de contexto para Claude.

In [1]:
# Install VoyageAI and Anthropic libraries
%pip install voyageai anthropic python-dotenv numpy

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 48 kB 2.0 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 5.3 MB/s eta 0:00:01
     |████████████████████████████████| 499 kB 84.6 MB/s eta 0:00:01
     |████████████████████████████████| 461 kB 21.6 MB/s eta 0:00:01
     |████████████████████████████████| 572 kB 14.9 MB/s eta 0:00:01
     |████████████████████████████████| 396 kB 25.5 MB/s eta 0:00:01
     |████████████████████████████████| 66 kB 11.2 MB/s eta 0:00:01
     |████████████████████████████████| 54 kB 8.0 MB/s  eta 0:00:01
     |████████████████████████████████| 640 kB 26.4 MB/s eta 0:00:01
     |████████████████████████████████| 245 kB 22.0 MB/s eta 0:00:01
     |████████████████████████████████| 625 kB 16.7 MB/s eta 0:00:01
     |████████████████████████████████| 80 kB 12.2 MB/s eta 0:00:01
     |████████████████████████████████| 56 kB 9.1 MB/s  eta 0:00:01
     |██████████████████████████

In [ ]:
# Client Setup
from dotenv import load_dotenv
import voyageai
import anthropic
import numpy as np

load_dotenv()

vo = voyageai.Client()
claude = anthropic.Anthropic()

In [ ]:
# Sample document & chunking by section
def chunk_by_section(document_text):
    sections = [s.strip() for s in document_text.split("\n## ") if s.strip()]
    chunks = []
    for i, sec in enumerate(sections):
        chunks.append(sec if i == 0 else f"## {sec}")
    return chunks

sample_doc = """# Architecture Overview
Our service processes forecast requests asynchronously using event queues and stateless workers.

## Authentication & Security
All API endpoints require short-lived JWT tokens signed with RS256 and validated at the API gateway.

## Rate Limiting & Retries
Clients must implement exponential backoff with jitter when receiving HTTP 429 Too Many Requests.

## Data Retention Policy
Audit logs are stored in immutable object storage for 365 days before automatic archival."""

chunks = chunk_by_section(sample_doc)
for idx, c in enumerate(chunks):
    print(f"--- Chunk {idx} ---\n{c}\n")

In [ ]:
# Generate embeddings for document chunks
def generate_embeddings(texts, model="voyage-3-large", input_type="document"):
    result = vo.embed(texts, model=model, input_type=input_type)
    return result.embeddings

chunk_embeddings = generate_embeddings(chunks, input_type="document")
print(f"Generated {len(chunk_embeddings)} embeddings of dimension {len(chunk_embeddings[0])}")

In [ ]:
# Cosine similarity & retrieval
def cosine_similarity(vec_a, vec_b):
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def retrieve_top_k(query, chunks, chunk_embeddings, k=2):
    query_embedding = generate_embeddings([query], input_type="query")[0]
    scored = [
        (cosine_similarity(query_embedding, emb), chunk)
        for emb, chunk in zip(chunk_embeddings, chunks)
    ]
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]

query = "How should clients handle HTTP 429 errors?"
top_chunks = retrieve_top_k(query, chunks, chunk_embeddings, k=2)
for score, text in top_chunks:
    print(f"Score: {score:.4f}\n{text}\n")

In [ ]:
# Grounded answer with Claude using retrieved context
context = "\n\n".join([text for _, text in top_chunks])

response = claude.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": f"Answer the question strictly using the provided context.\n\nContext:\n{context}\n\nQuestion: {query}"
        }
    ]
)

print(response.content[0].text)